# Publishing a Tool to the tool registry
This notebook showcases how to use the tool-registry REST API to publish a tool.

In [1]:
import requests
import json

In [12]:
#API_URL_BASE = "https://dev.tools-registry.eosc-data-commons.eu/api/v1/"
API_URL_BASE = "http://localhost:8080/api/v1/"
API_URL_TOOLS = f"{API_URL_BASE}tools/"
TOOL_REGISTRY_VERSION = "0.2.9"

## Getting access credentials
Before publishing to the tool registry an EGI access token is needed. This can be obtained from [https://aai.egi.eu/token/](https://aai.egi.eu/token/). 
Once obtained copy and paste in the cell below. The token is only valid for 1 hour.

In [13]:
TOKEN = ""

## Describing your tool
Next you want to describe your tool. The description should focus on what the tool does and what input data is needed.

### Minimum definition of a tool
- uri: A unique way to identify you tool e.g. "http://example.org/mytool/version/1".
- name: A name for the tool.
- version: The tool version.
- location: The source where the code or where the tools is defined e.g. Github repo.
- description: A description of what the tool does i.e. what are the main tasks of the tool.
- keywords: Keywords to catagorize your tool e.g. domain or specific task of a tool.
- tags: Additional tags that can help catagorize the tool e.g. organization.
- types: A list of types the tool can be considered part of e.g. Python Notebook, Galaxy Workflow.
- input_file_formats: A list of file formats expected by the tool e.g. csv, fasta.
- output_file_formats: A list of file outputs.
- input_file_descriptions: Instead of input file formats you can opt to describe the inputs.
- output_file_descriptions: Instead of output file formats you can opt to describe the inputs.
- raw_definition: If you tool has a specific definition like a workflow you can add it here.

In [16]:
tool = {
    "uri": "http://example.org/mytool/version/1",
    "name": "Example Tool",
    "version": "1",
    "location": "https://github.com/...",
    "description": "A good description",
    "keywords": ["", ""],
    "tags": ["example", ""],
    "license": "MIT",
    "types": ["example", ""],
    "input_file_formats": ["csv", "fasta", ""],
    "ouput_file_formats": [],
    "input_file_descriptions": ["", ""],
    "output_file_description": ["", ""],
    "raw_definition": {}
}   

## Check tool registry version
This notebook was created for tool-registry version `0.2.9`. Different versions might not have the same schema!

In [17]:
response = requests.get(f"{API_URL_BASE}health")
if response.status_code in (200, 201):
    data = response.json()
    version = data.get("version", None)
    if version and version == TOOL_REGISTRY_VERSION:
        print(f"Correct tool registry version: {data['version']}")
    else:
        print(f"Incorrect or missing version for tool registry: {response.text}")
else:
    print(f"Failure: {response.status_code}, message: {response.text}")

ConnectionError: HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /api/v1/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x13e3720d0>: Failed to establish a new connection: [Errno 61] Connection refused'))

## Publishing your tool
Next we can go ahead and submit the tool using a POST command.

In [ ]:
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {TOKEN}"
}
tool_id = None
response = requests.post(API_URL_TOOLS, json=tool, headers=headers)
if response.status_code in (200, 201):
    print("Success")
    tool_id = response.json()["tool_id"]
    print(response.text)
if response.status_code == 400:
    data = response.json()
    if data["detail"] and data["detail"]["existing_tool_id"]:
        tool_id = data["detail"]["existing_tool_id"]
        print(f"Tool already exists with ID: {tool_id}")

if not tool_id:
    print(f"Failure: {response.status_code}")
    print(response.text)


## Check your tool in the registry

On `Success`, the registry will respond with the tool ID. This is used to check/update/delete the tool in the future. 
Checking that your tool exists in the registry is a matter of calling the correct URL.

In [ ]:
if tool_id:
    
    tool_url = f"{API_URL_TOOLS}{tool_id}"
    tool_response = requests.get(tool_url)
    if tool_response.status_code in (200, 201):
        print(json.dumps(tool_response.json(), indent=4))
    else:
        print(f"Failure: {tool_response.status_code}")
        print(tool_response.text)
else:
    print("Tool was not created in previous step!")
    

## Update your tool information
It is also possible to update the tool inforamtion after being created. Updating tools is only allowed by the user that created the tool. This is done similary to creating a tool but instead we only provide the fields needing updating. 
In this example we will update the `name` and `license`

In [ ]:
if tool_id:
    tool_update = {
        "name": "New Example Tool",
        "license": "Apache 2",
    }
    tool_url = f"{API_URL_TOOLS}{tool_id}"
    tool_response = requests.patch(tool_url, json=tool_update, headers=headers)
    if tool_response.status_code in (200, 201):
        print(json.dumps(tool_response.json(), indent=4))
    else:
        print(f"Failure: {tool_response.status_code}")
        print(tool_response.text)
else:
    print("Tool was not created in previous step!")

## Delete your tool from registry

It is also possible to delete the tool from the registry. This is only allowed by the user that created the tool.

> ⚠️ **Warning:** Proceed with caution. This action will delete your tool from the registry.

In [ ]:
if tool_id:
    tool_url = f"{API_URL_TOOLS}{tool_id}"
    tool_response = requests.delete(tool_url, headers=headers)
    if tool_response.status_code in (200, 201):
        print(json.dumps(tool_response.json(), indent=4))
    else:
        print(f"Failure: {tool_response.status_code}")
        print(tool_response.text)
else:
    print("Tool was not created in previous step!")

## Verify tool has been deleted
Use the same query as before to check if the tool exists (it should not!)

In [ ]:
if tool_id:
    
    tool_url = f"{API_URL_TOOLS}{tool_id}"
    tool_response = requests.get(tool_url)
    if tool_response.status_code == 404:
        print(f"Tool {tool_id} not in registry.")
    else:
        print(f"Status: {tool_response.status_code}")
        print(tool_response.text)
else:
    print("Tool was not created in previous step!")